# 05 · 策略梯度:直接學「策略」

DQN 那一家是先學 **Q 值**,再從 Q 值推出動作。**策略梯度(Policy Gradient)** 走另一條路:**直接把策略 π(a|s) 參數化成一個神經網路**,輸出每個動作的機率,然後——

> **讓「帶來高報酬的動作」機率變大,「帶來低報酬的動作」機率變小。**

這一課手刻最基礎的策略梯度演算法 **REINFORCE**,在 CartPole 上跑。它是 PPO、A2C 這些現代演算法的共同祖先。

## 1. 安裝

In [ ]:
%pip install -q -U "gymnasium[classic-control]"

## 2. 策略網路

輸入狀態,輸出每個動作的機率(softmax)。用 `Categorical` 分布**取樣**動作(不是挑最大),取樣本身就是探索。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym

device = "cuda" if torch.cuda.is_available() else "cpu"


class PolicyNet(nn.Module):
    def __init__(self, n_obs, n_act):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, 128), nn.ReLU(),
            nn.Linear(128, n_act),
        )

    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)   # 動作機率分布

## 3. REINFORCE 訓練迴圈

做法:跑完**一整回合**,算出每一步的**折扣後報酬 return**,再用 `loss = −Σ log π(a|s) · return` 更新——報酬越高的動作,被推得越用力。報酬做標準化能讓訓練穩很多。

In [ ]:
env = gym.make("CartPole-v1")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

policy = PolicyNet(n_obs, n_act).to(device)
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)
gamma = 0.99
rewards_hist = []

for ep in range(400):
    s, _ = env.reset()
    log_probs, rewards = [], []
    done = False
    while not done:
        probs = policy(torch.tensor(s, dtype=torch.float32, device=device))
        dist = Categorical(probs)
        a = dist.sample()                       # 取樣動作
        log_probs.append(dist.log_prob(a))
        s, r, terminated, truncated, _ = env.step(int(a))
        rewards.append(r); done = terminated or truncated

    # 算每一步的折扣後 return(從後往前累加)
    returns, G = [], 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)   # 標準化

    loss = -(torch.stack(log_probs) * returns).sum()
    opt.zero_grad(); loss.backward(); opt.step()

    rewards_hist.append(sum(rewards))
    if ep % 40 == 0:
        print(f"ep {ep:3d}  reward {sum(rewards):5.0f}")
print("訓練完成。")

## 4. 學習曲線

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_rewards(rewards, window=50, title="Learning curve"):
    """畫每回合報酬與滑動平均，一眼看出 agent 有沒有在進步。"""
    plt.figure(figsize=(8, 4))
    plt.plot(rewards, alpha=0.3, label="episode reward")
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
        plt.plot(range(window - 1, len(rewards)), ma, lw=2.2,
                 label=f"moving avg ({window})")
    plt.xlabel("episode"); plt.ylabel("reward"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
plot_rewards(rewards_hist, window=20, title="REINFORCE on CartPole")

## 小結

- **策略梯度直接學策略** π(a|s),用取樣探索,不經過 Q 值。
- **REINFORCE**:跑完整回合 → 算折扣 return → `−Σ log π · return` 更新。
- **return 標準化**是讓它穩定的小撇步。
- PPO 就是在這個骨架上加了「不要一次更新太多」的護欄——你已經懂它的根。

下一課:當獎勵很稀疏時怎麼辦?認識 **gym wrappers 與獎勵塑形**。